# 09 — Step grading: does the practice grader find the first wrong step?

Practice problems ("Bài vận dụng") are solved in steps and graded against a rubric written with the problem. Submit
reports the checkpoints reached and **the first step that goes wrong**. A grader that flags a correct solution, or puts
the error in the wrong place, sends the student to fix the wrong thing, so this is the number to measure.

**Data.** ProcessBench (Zheng et al. 2024, *ProcessBench: Identifying Process Errors in Mathematical Reasoning*; Qwen;
Apache-2.0; see `datasets/README.md`). Each item is a model-written solution split into steps, with the index of the
earliest wrong step labelled by experts, or -1 when every step is right. The GSM8K split is used: 400 items, 207 with
an error.

**Method.** The app's own path, unchanged: a rubric is written as for a student's own problem
(`exercises.services.problems.own_rubric`), then the steps are graded (`exercises.services.grading.grade`). The model
calls run in the backend, `python manage.py eval_step_grading --n N`, which appends one line per item to
`ml_pipeline/reports/step_grading_gsm8k.jsonl` and skips items already graded. The free tier allows about twenty
requests a day per model and each item costs two, so the sample is grown over days; this notebook only reads that
file.

**Metrics** are ProcessBench's: accuracy on the solutions with an error (exact first wrong step), accuracy on the
correct ones (nothing flagged), and their harmonic mean (F1). Also reported: whether an error was flagged at all.

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().resolve().parents[1]))
from ml_pipeline.notebook_setup import bootstrap
bootstrap()
import pandas as pd
from ml_pipeline.step_grading import load, metrics, read_results
from IPython.display import display

items = {i['id']: i for i in load('gsm8k')}
rows = pd.DataFrame(read_results())
print(f"{len(rows)} items graded; {rows.pred.isna().sum()} refused as incomplete")
rows.head()

24 items graded; 2 refused as incomplete


,id,label,pred,n_steps,result
0,gsm8k-41,1,1.0,6,"{'verdict': 'needs_work', 'met': [], 'first_wr..."
1,gsm8k-394,-1,-1.0,4,"{'verdict': 'passed', 'met': [1, 2, 3], 'first..."
2,gsm8k-13,1,1.0,4,"{'verdict': 'needs_work', 'met': [], 'first_wr..."
3,gsm8k-372,-1,-1.0,3,"{'verdict': 'passed', 'met': [1, 2], 'first_wr..."
4,gsm8k-139,2,2.0,9,"{'verdict': 'needs_work', 'met': [], 'first_wr..."


## Refusals

Before grading, the rubric step decides whether a problem can be solved as written. A refusal is not a grade, so
refused items are left out of the metrics and counted on their own: every GSM8K problem is solvable, so each one is a
false refusal.

In [2]:
refused = rows[rows.pred.isna()]
for _, r in refused.iterrows():
    print(r.id, '|', items[r.id]['problem'][:160])
    print('   reason:', r.result.get('detail'))

gsm8k-29 | Wendy wants to place 20 more than double the number of books in a shelving system with 6 rows and 6 columns. How many books will she need to carry to complete h
   reason: The problem does not specify the initial number of books to be doubled.
gsm8k-83 | Lauren is saving 20% of every paycheck. How many more years does Lauren need to work if she plans to save for a retirement period of 20 years, live with 40% of 
   reason: The problem lacks information regarding the rate of return on savings or the inflation rate, which are necessary to calculate the accumulation and depletion of funds.


## Results

In [3]:
graded = rows[rows.pred.notna()].copy()
graded['pred'] = graded.pred.astype(int)
m = metrics(graded.label.tolist(), graded.pred.tolist())
pd.Series(m).to_frame('value')

,value
n,22.000000
n_with_error,10.000000
error_acc,0.800000
correct_acc,0.916667
f1,0.854369
detect_acc,0.954545


In [4]:
graded['has_error'] = graded.label != -1
graded['outcome'] = graded.apply(
    lambda r: 'right step' if r.has_error and r.pred == r.label
    else 'error missed' if r.has_error and r.pred == -1
    else 'wrong step' if r.has_error
    else 'correct, passed' if r.pred == -1 else 'false alarm', axis=1)
graded.groupby(['has_error', 'outcome']).size().to_frame('items')

items
has_error outcome               
False     correct, passed     11
          false alarm          1
True      right step           8
          wrong step           2

## Where it goes wrong

Each miss with the step the experts marked and the step the grader picked, to tell a grader that is a step early or
late (often a matter of where an error is said to *start*) from one that is simply wrong.

In [5]:
misses = graded[graded.label != graded.pred]
for _, r in misses.iterrows():
    item = items[r.id]
    print(f"{r.id}: gold step {r.label}, graded step {r.pred} ({r.result.get('error_type')})")
    print('   ', (r.result.get('detail') or '')[:200])
if misses.empty:
    print('No misses in this sample.')

gsm8k-387: gold step -1, graded step 1 (misread)
    The problem asks for the number of vines Steve needs for his own consumption, not for the combined consumption of both Steve and his girlfriend.
gsm8k-68: gold step 2, graded step 3 (concept)
    The total cost for the plan is the sum of the main phone's cost and the costs of all additional phones, not five times the main phone's cost.
gsm8k-312: gold step 0, graded step 1 (missing_step)
    The attempt calculates the total time directly from the total distance, omitting the intermediate calculations for the time taken for each segment of James's journey.


## Outside maths: the hand-made set

No public set labels the first wrong step outside maths, so `datasets/handmade/practice_steps.json` was written for
this project: 24 problems in four subjects, two in English (programming, contract law) and two in Vietnamese
(economics, biology). Each has a model solution and two more attempts, either wrong on purpose (first wrong step
and error type known by construction) or **right by another route**: a shortcut or a different argument. Those
check the weakness seen above, a grader that calls a short correct answer "missing a step". A label equal to the
number of steps means the attempt stops short.

The labels were set by construction and still want a person's review. Run with
`python manage.py eval_step_grading --split handmade`; it costs one request per problem for the rubric and one per
attempt, so the set fills in over a few days.

In [6]:
hm = pd.DataFrame(read_results('handmade'))
print(f"{len(hm)} of 72 attempts graded")
if len(hm):
    hm_graded = hm[hm.pred.notna()].copy()
    hm_graded['pred'] = hm_graded.pred.astype(int)
    print(f"{len(hm) - len(hm_graded)} refused")
    overall = pd.Series(metrics(hm_graded.label.tolist(), hm_graded.pred.tolist()), name='all')
    by = {k: pd.Series(metrics(g.label.tolist(), g.pred.tolist()), name=k) for k, g in hm_graded.groupby('subject')}
    display(pd.concat([overall, *by.values()], axis=1).T[['n', 'error_acc', 'correct_acc', 'f1', 'detect_acc']])

72 of 72 attempts graded
0 refused


,n,error_acc,correct_acc,f1,detect_acc
all,72.0,0.976744,0.827586,0.896000,0.930556
contract law,18.0,1.000000,0.625000,0.769231,0.833333
kinh tế,18.0,1.000000,1.000000,1.000000,1.000000
programming,18.0,1.000000,0.714286,0.833333,0.888889
sinh học,18.0,0.909091,1.000000,0.952381,1.000000


In [7]:
# Right by another route: the shortcut and different-argument attempts (label -1, not the model solution).
if len(hm):
    alt = hm_graded[(hm_graded.label == -1) & ~hm_graded.id.str.endswith('/0')]
    print(f"other correct routes passed: {(alt.pred == -1).sum()} of {len(alt)}")
    for _, r in alt[alt.pred != -1].iterrows():
        print('  flagged:', r.id, '|', r.result.get('error_type'), '|', (r.result.get('detail') or '')[:160])

other correct routes passed: 2 of 5
  flagged: code-2/1 | missing_step | The explanation does not state that binary search divides the list in half with each comparison.
  flagged: law-2/1 | missing_step | The response fails to explain why the shop can refuse the sale, specifically by omitting the requirement that the shop is entitled to correct the error before t
  flagged: law-6/2 | missing_step | The response identifies the act as consideration but fails to explain that legal forbearance constitutes valid consideration for a contract.


In [8]:
# Where the step was found, was the kind of error named right?
if len(hm):
    found = hm_graded[(hm_graded.label != -1) & (hm_graded.label == hm_graded.pred)].copy()
    found['named'] = found.result.map(lambda r: r.get('error_type'))
    print(f"error type matches on {(found.named == found.error_type).sum()} of {len(found)} correctly placed errors")
    display(pd.crosstab(found.error_type, found.named))

error type matches on 30 of 42 correctly placed errors


named,concept,execution,misread,missing_step
error_type,,,,
concept,19,0,0,0
execution,2,6,0,0
misread,4,1,2,0
missing_step,0,1,0,3
unsupported,4,0,0,0


## Reading the numbers

- The sample is small and grown a few items a day, so treat the figures as a first check, not a benchmark score. For
  scale: ProcessBench reports F1 on GSM8K from about 30 for small open models to the high 80s for the strongest
  critics.
- Two calls per item mean the grader is measured together with the rubric it is given, as in the app.
- GSM8K is maths. The hand-made set above covers four other subjects, but it is small and written for this project;
  a set written and labelled by teachers of those subjects would be the stronger check.